In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset 08 — Breast Cancer Wisconsin (Original) — UCI

**Descripción:** Dataset clínico para diagnóstico de cáncer de mama. Las características se calcularon a partir de imágenes digitalizadas de aspirado con aguja fina (FNA) de masas mamarias. Describe características del núcleo celular.

**Tarea:** Clasificación binaria — predecir si el tumor es **Benigno (2)** o **Maligno (4)**.

**Características:**
- m = 699 registros (con 16 con valores nulos)
- n = 9 características + ID + clase
- Variable objetivo: `Class` (2 = benigno, 4 = maligno)
- Valores nulos: 16 registros con `?` en `Bare_Nuclei`
- Variables: Clump Thickness, Cell Size, Cell Shape, Marginal Adhesion, etc.

**URL UCI:** https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(original)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
print('Librerías cargadas correctamente')

## 1. Carga del Dataset

In [ ]:
columnas = ['ID','Clump_Thickness','Uniformity_Cell_Size','Uniformity_Cell_Shape',
           'Marginal_Adhesion','Single_Epithelial_Cell_Size','Bare_Nuclei',
           'Bland_Chromatin','Normal_Nucleoli','Mitoses','Class']
ruta = '/content/drive/MyDrive/machine learning/datasets/breast-cancer-wisconsin.data'
data = pd.read_csv(ruta, names=columnas, na_values='?')
print(f'Dimensiones: {data.shape}')
data.head()

## 2. Revisión de Tipos y Nulos

In [ ]:
print(data.dtypes)
print()
nulos = data.isnull().sum()
print('Valores nulos:')
print(nulos[nulos>0])

## 3. Preprocesamiento con Pandas

In [ ]:
df = data.copy()
df = df.drop(columns=['ID'])  # ID no es predictivo

# Imputar nulos con mediana
df['Bare_Nuclei'] = df['Bare_Nuclei'].fillna(df['Bare_Nuclei'].median())

# Codificar objetivo: 2=Benigno→0, 4=Maligno→1
df['target'] = (df['Class'] == 4).astype(int)

X = df[[col for col in df.columns if col not in ['Class','target']]].to_numpy(dtype=np.float64)
y = df['target'].to_numpy()

print(f'X: {X.shape} | y: {y.shape}')
print(f'Benigno  (0): {int((y==0).sum())} ({(y==0).mean()*100:.1f}%)')
print(f'Maligno  (1): {int((y==1).sum())} ({(y==1).mean()*100:.1f}%)')

## 4. Estadísticas Descriptivas

In [ ]:
df.describe()

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].bar(['Benigno (0)','Maligno (1)'],[int((y==0).sum()),int((y==1).sum())],
            color=['steelblue','tomato'],edgecolor='black')
axes[0].set_title('Distribución de Diagnóstico')
axes[0].set_ylabel('Número de casos')

axes[1].hist(df['Clump_Thickness'],bins=10,color='steelblue',edgecolor='black')
axes[1].set_title('Clump Thickness')

for val,color,label in [(0,'steelblue','Benigno'),(1,'tomato','Maligno')]:
    subset = df[df['target']==val]['Uniformity_Cell_Size']
    axes[2].hist(subset,bins=10,alpha=0.6,color=color,label=label,edgecolor='black')
axes[2].set_title('Uniformidad Tamaño Celular por Diagnóstico')
axes[2].legend()
plt.tight_layout()
plt.show()

## 6. División 80/20

In [ ]:
np.random.seed(42)
idx = np.random.permutation(len(X))
corte = int(len(X)*0.8)
X_train,X_test = X[idx[:corte]],X[idx[corte:]]
y_train,y_test = y[idx[:corte]],y[idx[corte:]]
print(f'Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}')

## 7. Conclusiones

**Breast Cancer Wisconsin (Original)** es un dataset médico clásico con 699 registros y 9 características de biopsias mamarias. Solo tiene 16 valores nulos en `Bare_Nuclei` que se imputan con mediana. La distribución de clases es: 65.5% benigno y 34.5% maligno. Es ideal para clasificación binaria con regresión logística. Alta precisión esperada ya que las características son discriminativas.